## Imports necessários

In [4]:
from pymongo import MongoClient
import pymongoarrow
from pymongoarrow.api import Schema
import pandas
from pymongoarrow.monkey import patch_all
from datetime import datetime
import pyarrow as pa
patch_all()


In [ ]:
import numpy
print(numpy.__version__)

## Conexão com o mongodb

In [6]:
client = MongoClient("mongodb://mongo1:27017,mongo2:27017,mongo3:27017/loja?replicaSet=db-replica-set&readPreference=secondaryPreferred&readPreferenceTags=dc:SP")

In [ ]:
client.list_database_names()

### Configuração do banco de dados e collection

In [8]:
db = client["loja"]
colecao = db["produtos"]


## Convertendo para o Pandas os documentos do Mongodb

In [10]:
df= colecao.find_pandas_all({"nomeProduto": "Produto 19"})

In [ ]:
print(df.head())

## Convertendo para o Apache Arrow os documentos do Mongodb

In [12]:
arrow_table = colecao.find_arrow_all({"nomeProduto": "Produto 19"})

In [ ]:
print(arrow_table)

## Configurando o schema para o resultado

In [20]:
# Definir um Schema para os produtos
produto_schema = Schema({
    "idProduto": pa.int32(),
    "nomeProduto": pa.string(),
    "valorProduto": pa.float64(),
    "idSkus": pa.list_(pa.int32()),  # Array de inteiros
    "skus": pa.list_(pa.struct({     # Array de JSONs embutidos
        "idSku": pa.int32(),
        "nome": pa.string(),
        "valor": pa.float64()
    })),
     "categorias": pa.list_(pa.struct({     # Array de JSONs embutidos
        "idCategoria": pa.int32(),
        "nome": pa.string()     
    })),  # Array de strings
     "marcas": pa.list_(pa.struct({     # Array de JSONs embutidos
        "idMarca": pa.int32(),
        "nome": pa.string()     
    })),  # Array de strings
    "imagens": pa.list_(pa.struct({   # Array de JSONs para imagens
        "url": pa.string()      
    }))
})


In [21]:
df= colecao.find_pandas_all({"nomeProduto": "Produto 19"},schema=produto_schema)

In [ ]:
# Exibir a estrutura do schema retornado
print(df)

## Consultando as informações com agregações

In [23]:
df = colecao.aggregate_pandas_all([{ "$unwind": "$skus" },{ "$project": {"_id":0, "nomeProduto": 1, "idSku": "$skus.idSku", "nomeSku": "$skus.nome", "valor": "$skus.valor" } }])

In [ ]:
print(df)

## Escrevendo o resultado em uma coleção do Mongodb

In [ ]:
from pymongoarrow.api import write
colecao_sku = db["skus"]
write(colecao_sku, df)